In [ ]:
import requests
from typing import List, Dict, Any
import os
from pathlib import Path
import base64
import ujson as json
from dotenv import load_dotenv
import uuid

from langfuse.decorators import observe, langfuse_context
from langfuse import Langfuse
import re

_ = load_dotenv("../.env.dev")

class LLMClient:
    def __init__(self, host: str = None, model: str = None, api_key: str = None):
        self.host = os.getenv("LLM_HOST", host)
        self.model = os.getenv("LLM_MODEL", model)
        self.api_key = os.getenv("LLM_API_KEY", api_key)


    @observe(as_type="generation")
    def completions(
            self,
            temperature: int = 0.01,
            messages: List[Dict[str, Any]] = None,
            **kwargs):


        if messages is None:
            messages = []


        payload = {
            "model": self.model,
            "temperature": temperature,
            "messages": messages
        }
        headers = {
            "Content-Type": "application/json"
        }

        headers = headers | {"Authorization": f"Bearer {self.api_key}"} if self.api_key else headers

        langfuse_context.update_current_observation(
            input=messages,
            model=self.model,
            metadata=payload
        )

        response = requests.post(
            f"{self.host}/v1/chat/completions",
            headers=headers,
            data=json.dumps(payload)
        )


        if response.status_code == 200:
            completion_token = response.json()["usage"]["completion_tokens"]
            total_token = response.json()["usage"]["total_tokens"]
            input_token = total_token - completion_token
            langfuse_context.update_current_observation(
            output=response.json()["choices"][0]["message"]["content"],
            usage_details={
                "input": input_token,
                "output": completion_token,
            },
            prompt=kwargs.get("prompt"),
        )
            return {
                "data": response.json()["choices"][0]["message"]["content"],
                "status_code": response.status_code
                }
        else:
            return {
                "data": None,
                "status_code": response.status_code,



 

In [ ]:
# import requests
# import json
 
# url = "http://10.1.87.71:30042/v1/chat/completions"
 
# payload = json.dumps({
#     "temperature": 0,
#     "top_p": 0.1,
#     "stream": False,
#     "frequency_penalty": 0,
#     "model": "model/beegen-model-v1",
#     "messages": [
#         {
#         "role": "user",
#         "content": "xin chào"
#         }
#     ]
# })
# headers = {
#   'Content-Type': 'application/json',
#   'Authorization': 'Bearer mb123456789',
# }
 
# response = requests.request("POST", url, headers=headers, data=payload)
 
# print(response.text)

In [10]:
import os
import json
import requests
import uuid
from typing import List, Dict, Any
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from dotenv import load_dotenv



class ChatAPI:
    def __init__(self, langfuse_config=None):
        """Initialize the ChatAPI with Langfuse configuration."""
        load_dotenv()
        
        # Use provided config or load default
        if langfuse_config is None:
            try:
                langfuse_config = LangfuseConfig.default()
            except Exception as e:
                print(f"Error loading default config: {str(e)}")
                # Fallback to environment variables
                self.langfuse = Langfuse(
                    secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
                    public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
                    host=os.getenv('LANGFUSE_HOST')
                )
        else:
            # Use the provided config
            self.langfuse = Langfuse(
                secret_key=langfuse_config.secret_key,
                public_key=langfuse_config.public_key,
                host=langfuse_config.host,
                debug=langfuse_config.debug
            )
            
            # Set no_proxy if needed
            if hasattr(langfuse_config, 'no_proxy') and langfuse_config.no_proxy:
                os.environ['no_proxy'] = langfuse_config.no_proxy
        
        # API configuration
        self.url = os.getenv('LLM_HOST', "http://10.1.87.71:30042/v1/chat/completions")
        self.model = os.getenv('LLM_MODEL', "model/beegen-model-v1")
        self.api_key = os.getenv('LLM_API_KEY', "mb123456789")
        
        self.headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {self.api_key}',
        }
        
        # Create a trace ID for linking operations
        self.trace_id = str(uuid.uuid4())

    @observe(as_type="generation")
    def chat_completion(
            self,
            messages: List[Dict[str, Any]],
            temperature: float = 0,
            top_p: float = 0.1,
            frequency_penalty: float = 0,
            stream: bool = False,
            **kwargs
    ) -> Dict:
        """
        Send a chat completion request with Langfuse tracing.
        This method is decorated with @observe to automatically create a trace.
        """
        # Build payload
        payload = {
            "temperature": temperature,
            "top_p": top_p,
            "stream": stream,
            "frequency_penalty": frequency_penalty,
            "model": self.model,
            "messages": messages
        }

        # Add any additional parameters from kwargs
        for key, value in kwargs.items():
            if key not in payload:
                payload[key] = value

        # Update the current observation with input data
        langfuse_context.update_current_observation(
            name="chat-completion",
            input=messages,
            model=self.model,
            metadata=payload
        )

        try:
            response = requests.post(
                self.url, 
                headers=self.headers, 
                json=payload
            )
            response.raise_for_status()
            response_data = response.json()

            # Extract usage information
            completion_tokens = response_data.get("usage", {}).get("completion_tokens", 0)
            total_tokens = response_data.get("usage", {}).get("total_tokens", 0)
            input_tokens = total_tokens - completion_tokens

            # Update the observation with output data
            langfuse_context.update_current_observation(
                output=response_data.get("choices", [{}])[0].get("message", {}).get("content"),
                usage_details={
                    "input": input_tokens,
                    "output": completion_tokens,
                    "total": total_tokens
                },
                prompt=kwargs.get("prompt"),
            )

            return {
                "data": response_data.get("choices", [{}])[0].get("message", {}).get("content"),
                "status_code": response.status_code,
                "usage": {
                    "input_tokens": input_tokens,
                    "output_tokens": completion_tokens,
                    "total_tokens": total_tokens
                },
                "raw_response": response_data
            }

        except Exception as e:
            error_msg = str(e)
            print(f"API request failed: {error_msg}")
            
            # Update observation with error info
            langfuse_context.update_current_observation(
                status="error",
                error=error_msg
            )
            
            return {
                "data": None,
                "status_code": getattr(response, 'status_code', 500),
                "error": error_msg
            }

    def send_chat_completion(self, prompt: str) -> dict:
        """Simplified method to send a single prompt message"""
        return self.chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

    def cleanup(self):
        """Cleanup Langfuse resources"""
        self.langfuse.flush()


def main():
    """Main function to run the chat API test"""
    try:
        # Try to load config from LangfuseConfig
        try:
            from beegen.configs import LangfuseConfig
            langfuse_config = LangfuseConfig.default()
            chat_api = ChatAPI(langfuse_config)
        except Exception as e:
            print(f"Error loading LangfuseConfig: {str(e)}")
            chat_api = ChatAPI()  # Fallback to environment variables
            
        print("Initializing chat API test...")
        
        try:
            # Simple test
            result = chat_api.send_chat_completion("xin chào")
            print(f"Received response: {result['data']}")
            
            # More complex test with conversation
            conversation_result = chat_api.chat_completion(
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "Tell me about MB Bank."}
                ],
                temperature=0.2
            )
            print(f"Conversation response: {conversation_result['data']}")
            print(f"Token usage: {conversation_result['usage']}")
            
        except Exception as e:
            print(f"Error in chat completion: {str(e)}")
            raise
            
        finally:
            chat_api.cleanup()
            
    except Exception as e:
        print(f"Error in main: {str(e)}")
        raise


if __name__ == "__main__":
    main()

Error loading LangfuseConfig: No module named 'beegen'
Error loading default config: name 'LangfuseConfig' is not defined
Initializing chat API test...
Received response: Xin chào! Rất vui được gặp bạn. Tôi có thể giúp gì cho bạn hôm nay?
Conversation response: MB Bank, also known as Military Commercial Joint Stock Bank, is one of the largest and most reputable commercial banks in Vietnam. Here are some key points about MB Bank:

1. History: Founded in 1994, MB Bank has grown significantly over the past three decades.

2. Ownership: The bank is a joint-stock company, with the Vietnam Military Economics Corporation (VMEC) as its largest shareholder.

3. Services: MB Bank offers a wide range of financial services, including:
   - Retail banking
   - Corporate banking
   - Investment banking
   - Insurance
   - Asset management
   - Digital banking

4. Network: The bank has a vast network of branches and ATMs across Vietnam, with over 500 branches and more than 2,000 ATMs.

5. Digital tra